# Coding practice: audit calibration under shift

The same model's predictions on ten examples like its training data, and on ten from a new site. Measure how well its confidence matches how often it is right in each, then choose a decision threshold for the new site that keeps recall high enough. No downloads or GPU are needed.

> Save your own copy first: File → Save a copy in Drive.

In [ ]:
import numpy as np

IID_LABELS = np.array([1, 1, 1, 1, 0, 0, 0, 0, 1, 0])
IID_PROBS  = np.array([.95, .88, .77, .62, .51, .42, .31, .12, .83, .18])
SITE_LABELS = np.array([1, 1, 1, 1, 0, 0, 0, 0, 1, 0])
SITE_PROBS  = np.array([.82, .66, .54, .48, .71, .58, .44, .22, .61, .35])


## Task 1: a two-bin calibration error

Using the bins `[0, 0.5)` and `[0.5, 1]`, compute the expected calibration error. Empty bins contribute zero.

<details style="border:1px solid #e5e7eb;border-radius:8px;padding:10px 14px;background:#f9fafb;color:#111827;margin:14px 0;">
<summary style="cursor:pointer;font-weight:600;">Hint: how the calibration error is defined</summary>

Take the weighted average over the bins of `abs(bin accuracy - bin mean confidence)`, weighting each bin by its share of the examples.

</details>

In [ ]:
def two_bin_ece(labels, probabilities):
    total = len(labels)
    ece = 0.0
    for lower, upper in ((0.0, 0.5), (0.5, 1.000001)):
        mask = (probabilities >= lower) & (probabilities < upper)
        if mask.any():
            # TODO 1: add this bin's contribution to the calibration error.
            raise NotImplementedError
    return ece


## Task 2: a threshold under a recall constraint

Among the supplied thresholds, return the largest one whose recall is at least the required value. That is the most selective operating point that still meets the requirement.

In [ ]:
def recall_at_threshold(labels, probabilities, threshold):
    predictions = probabilities >= threshold
    true_positives = ((predictions == 1) & (labels == 1)).sum()
    positives = (labels == 1).sum()
    return true_positives / positives


def choose_threshold(labels, probabilities, thresholds, minimum_recall):
    # TODO 2: filter feasible thresholds and return the largest one.
    raise NotImplementedError


In [ ]:
# Two toy cases with answers you can check by hand.
assert abs(two_bin_ece(np.array([1, 0]), np.array([1.0, 0.0]))) < 1e-12
assert abs(two_bin_ece(np.array([0, 0]), np.array([0.9, 0.9])) - 0.9) < 1e-12

iid_ece = two_bin_ece(IID_LABELS, IID_PROBS)
site_ece = two_bin_ece(SITE_LABELS, SITE_PROBS)
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
site_threshold = choose_threshold(SITE_LABELS, SITE_PROBS, thresholds, 0.8)

# The chosen threshold must meet the recall requirement, and no larger one may.
assert recall_at_threshold(SITE_LABELS, SITE_PROBS, site_threshold) >= 0.8
assert all(recall_at_threshold(SITE_LABELS, SITE_PROBS, t) < 0.8
           for t in thresholds if t > site_threshold)
print({"iid_ece": round(iid_ece, 3), "site_ece": round(site_ece, 3),
       "site_threshold_for_recall_at_least_0.8": site_threshold})

## Interpret before acting

In your copy, write one claim comparing calibration on the two sets, the numbers that are your evidence, one reason to doubt the comparison, and what you would want to have before trusting this model at the new site.